# BB84 Quantum Key Distribution

Alice encodes random bits in random bases (Z or X), Bob measures in random
bases. After public comparison, they keep matching bases and check QBER
for eavesdropping.

In [ ]:
import random
import qiskit as qk
import qiskit_aer as qka

NUM_BITS = 16

## Encode and measure

In [ ]:
def encode_bit(bit, basis):
    qc = qk.QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == "+":
        qc.h(0)
    return qc


def measure_bit(qc, basis):
    if basis == "+":
        qc.h(0)
    qc.measure(0, 0)
    backend = qka.AerSimulator()
    compiled = qk.transpile(qc, backend)
    counts = backend.run(compiled, shots=1).result().get_counts()
    return int(list(counts.keys())[0], 2)

## Run BB84 without eavesdropper

In [ ]:
def run_bb84(eavesdrop=False):
    alice_bits = [random.randint(0, 1) for _ in range(NUM_BITS)]
    alice_bases = [random.choice(["z", "+"]) for _ in range(NUM_BITS)]
    bob_bases = [random.choice(["z", "+"]) for _ in range(NUM_BITS)]

    print(f"Alice bases: {' '.join(alice_bases)}")
    print(f"Bob bases:   {' '.join(bob_bases)}")

    key_alice, key_bob = [], []
    for i in range(NUM_BITS):
        qc = encode_bit(alice_bits[i], alice_bases[i])
        if eavesdrop and random.random() < 0.5:
            measure_bit(qc, random.choice(["z", "+"]))
        result = measure_bit(qc, bob_bases[i])
        key_bob.append(result)
        if alice_bases[i] == bob_bases[i]:
            key_alice.append(alice_bits[i])

    sift_a = [alice_bits[i] for i in range(NUM_BITS) if alice_bases[i] == bob_bases[i]]
    sift_b = [key_bob[i] for i in range(NUM_BITS) if alice_bases[i] == bob_bases[i]]
    print(f"Sifted key A: {sift_a}")
    print(f"Sifted key B: {sift_b}")

    check = random.sample(range(len(sift_a)), min(4, len(sift_a)))
    errors = sum(1 for i in check if sift_a[i] != sift_b[i])
    qber = errors / len(check)
    print(f"QBER: {errors}/{len(check)} = {qber:.2%}")
    if qber > 0.11:
        print("EAVESDROPPING DETECTED!")
    else:
        final = [sift_a[i] for i in range(len(sift_a)) if i not in check]
        print(f"Final key: {final}")


print("=== No eavesdropper ===")
run_bb84(eavesdrop=False)
print("\n=== With eavesdropper ===")
run_bb84(eavesdrop=True)